<a href="https://colab.research.google.com/github/sparshbansal-newton/deep-learning-labs/blob/main/Notebooks/6_Auto_grad/pytorch_autograd_Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch Autograd

**Autograd** is PyTorch's engine for computing gradients automatically. It records every operation on tensors with `requires_grad=True` into a **computational graph**, then walks it backward via `.backward()` to apply the chain rule — this is **backpropagation**.

**Covered here:**
1. `.backward()` vs. manual differentiation
2. Chain rule through `sin(x²)`
3. BCE loss gradients for logistic regression — by hand, then verified
4. Vector gradients & accumulation
5. Clearing gradients with `.zero_()`
6. Disabling tracking: `requires_grad_(False)`, `.detach()`, `no_grad()`

As you run each cell, watch `grad_fn=<...>` appear on tracked tensors and vanish when tracking is off.

### Manual differentiation

For $y = x^2$, $\frac{dy}{dx} = 2x$. At $x=3$, that's $6$.

Fine for one line — but a real loss composes thousands of operations over millions of parameters. Hand-coding each gradient doesn't scale. That's what autograd solves.

In [3]:
def dy_dx(x):
  return x*2

In [4]:
dy_dx(3)

6

## 1. Autograd Basics

Create a tensor with `requires_grad=True` and PyTorch records every operation on it, building a graph you walk backward with `.backward()`.

In [ ]:
import torch

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

In [ ]:
y = x**2

In [ ]:
x

In [ ]:
y

- `requires_grad=True` — track operations on `x`
- `y = x**2` — recorded; `grad_fn=<PowBackward0>` is the recipe for $dy/dx$
- `y.backward()` — walks the graph backward
- `x.grad` → `6.0`, matching `dy_dx(3)`

In [ ]:
y.backward()

In [ ]:
x.grad

## 2. Chain Rule

Two-step composition: $y = x^2$, $z = \sin(y)$.

$$\frac{dz}{dx} = \cos(x^2)\cdot 2x$$

Autograd does exactly this for any chain: each `grad_fn` knows its local derivative, and `.backward()` multiplies them along the graph.

In [ ]:
import math

def dz_dx(x):
    return 2 * x * math.cos(x**2)

In [ ]:
dz_dx(4)

In [ ]:
x = torch.tensor(4.0, requires_grad=True)

In [ ]:
y = x ** 2

In [ ]:
z = torch.sin(y)

In [ ]:
x

In [ ]:
y

In [ ]:
z

In [ ]:
z.backward()

In [ ]:
x.grad

**The `UserWarning`:** PyTorch keeps `.grad` only for **leaf tensors** (like `x`). Intermediates like `y` are freed after `backward()` to save memory. Need it? Call `y.retain_grad()` before `backward()`.

In [ ]:
y.grad

## 3. Logistic Regression Gradients

Compute **Binary Cross-Entropy** gradients w.r.t. `w` and `b` — by hand, then with autograd.

Setup: input $x$, label $y \in \{0,1\}$, prediction $\hat{y} = \sigma(wx + b)$, loss $L = -\big(y\log\hat{y} + (1-y)\log(1-\hat{y})\big)$.

In [ ]:
import torch

# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

**Chain rule**, with $z = wx+b$:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z}\cdot\frac{\partial z}{\partial w}, \qquad \frac{\partial L}{\partial b} = \frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z}\cdot\frac{\partial z}{\partial b}$$

Each factor is a standard derivative (BCE, sigmoid, linear). Multiplying them = backprop by hand, for one neuron.

In [ ]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [ ]:
# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [ ]:
loss

In [ ]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [ ]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

In [ ]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [ ]:
w

In [ ]:
b

In [ ]:
z = w*x + b
z

In [ ]:
y_pred = torch.sigmoid(z)
y_pred

In [ ]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

In [ ]:
loss.backward()

In [ ]:
print(w.grad)
print(b.grad)

**Same numbers, via autograd.** `w.grad` and `b.grad` match the hand-derived `dL_dw` and `dL_db`. You write the forward pass; autograd derives the backward pass — at any depth.

## 4. Vector Gradients

Autograd works the same for tensors. The only rule: `.backward()` needs a **scalar** output, so we reduce with `.mean()` first.

For $y = \frac{1}{3}\sum x_i^2$, $\frac{\partial y}{\partial x_i} = \frac{2x_i}{3}$ — exactly the `x.grad` values below (`[0.667, 1.333, 2.000]` for `x = [1, 2, 3]`).

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

In [ ]:
x

In [ ]:
y = (x**2).mean()
y

In [ ]:
y.backward()

In [ ]:
x.grad

## 5. Clearing Gradients

PyTorch **accumulates** gradients — each `.backward()` *adds* to `.grad`. Useful across mini-batches, but in a training loop you must zero them first (`optimizer.zero_grad()`), or old gradients leak into the new step. Here, `x.grad.zero_()` resets the accumulated gradient back to `0`.

In [ ]:
# clearing grad
x = torch.tensor(2.0, requires_grad=True)
x

In [ ]:
y = x ** 2
y

In [ ]:
y.backward()

In [ ]:
x.grad

In [ ]:
x.grad.zero_()

## 6. Disabling Gradient Tracking

At inference, tracking just wastes memory. Three ways to turn it off:

1. **`x.requires_grad_(False)`** — in-place, permanent
2. **`x.detach()`** — new tensor sharing data, detached from the graph; `x` untouched
3. **`torch.no_grad()`** — context manager scoping off tracking for its block

Below we contrast the first two — watch `grad_fn` disappear once tracking is off.

In [ ]:
# disable gradient tracking
x = torch.tensor(2.0, requires_grad=True)
x

In [ ]:
y = x ** 2
y

In [ ]:
y.backward()

In [ ]:
x.grad

In [ ]:
# option 1 - requires_grad_(False)
# option 2 - detach()
# option 3 - torch.no_grad()

In [ ]:
x.requires_grad_(False)

**What just happened:** while `x` was tracked, `y = x**2` carried a `grad_fn` and `y.backward()` filled `x.grad` (`4.0`). After `x.requires_grad_(False)`, tracking is off **permanently** — rebuilding `y = x**2` now gives a plain tensor with **no `grad_fn`** (see below), so there is no graph to walk. Calling `.backward()` on it would raise `RuntimeError: element 0 ... does not require grad`.

In [ ]:
x

In [ ]:
y = x ** 2

In [ ]:
y

`detach()` is the non-destructive option: `x` keeps `requires_grad=True` and still backprops, while `z = x.detach()` is a gradient-free snapshot of the same value.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
x

In [ ]:
z = x.detach()
z

In [ ]:
y = x ** 2

In [ ]:
y

In [ ]:
y1 = z ** 2
y1

In [ ]:
y.backward()

**The contrast:** `y = x**2` (from tracked `x`) carries a `grad_fn` and `y.backward()` runs fine; `y1 = z**2` (from detached `z`) has no `grad_fn`. That is the point of `detach()` — branch off a value without breaking gradient flow through `x`. The cells below are a final quick recap of the basic track → `backward()` → `.grad` loop.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
x

In [ ]:
y = x ** 2

In [ ]:
y

In [ ]:
y.backward()

## Conclusion

- **Track**: `requires_grad=True` records ops into a graph (see `grad_fn`)
- **Backward**: `.backward()` on a scalar applies the chain rule in reverse
- **Store**: results land in `.grad` (leaf tensors only) and **accumulate** — `.zero_()` between steps
- **Turn off**: `requires_grad_(False)`, `.detach()`, `torch.no_grad()`
- **Turn on**: `.retain_grad()`

The logistic-regression example is the takeaway: hand-derived $\partial L/\partial w$ = `w.grad`, exactly. Autograd isn't magic — it's the same calculus, automated for graphs too large to differentiate by hand.

